In [1]:
import numpy as np 
import pandas as pd

movies=pd.read_csv('final.csv')
movies[movies.duplicated(subset='id', keep=False)]
movies.drop_duplicates(subset='id', keep='first', inplace=True)

In [2]:

from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

def stem(text):
    l=[]
    for i in text.split():
        l.append(ps.stem(i))
    string=" ".join(l)
    return string

In [3]:
cast=movies[['id','cast']]
cast=cast.copy()
cast['cast'] = cast['cast'].fillna('[]')
import ast

cast.loc[:, 'cast'] = cast['cast'].fillna('[]')

# Step 1: Convert cast strings to actual lists of tuples
cast.loc[:, 'cast'] = cast['cast'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Step 2: Expand and convert to space-separated string
def expand_cast_to_string(cast_list):
    if not isinstance(cast_list, list):
        return ''
    expanded = [name for name, count in cast_list if isinstance(name, str) and isinstance(count, int) for _ in range(count)]
    return ' '.join(expanded)

# Step 3: Apply the function
cast.loc[:, 'cast'] = cast['cast'].apply(expand_cast_to_string)
cast.loc[:, 'cast'] = cast['cast'].str.replace('-', '', regex=False)



In [4]:
from sklearn.feature_extraction.text import CountVectorizer

cv_cast=CountVectorizer(stop_words='english')
cast_ind=cv_cast.fit_transform(cast['cast'])

cast_count=cast_ind.toarray().sum(axis=0)
cast_names=cv_cast.get_feature_names_out()
cast_freq=list(zip(cast_names,cast_count))

import os
os.makedirs('cast',exist_ok=True)

top_cast=sorted(cast_freq, key= lambda x:x[1],reverse=True)[:200]
df_top = pd.DataFrame(top_cast, columns=['cast', 'count'])
df_top['rank'] = df_top.index
df_top[['rank', 'cast']].to_csv('cast/top_200_cast.csv', index=False)

In [5]:
top_cast=df_top['cast'].tolist()

cv_top=CountVectorizer(vocabulary=top_cast)
top_ind=cv_top.fit_transform(cast['cast'])

top_dense=top_ind.toarray()
np.savez_compressed('cast/movie_cast_mapping.npz',top_dense)


#To read the array
data = np.load('cast/movie_cast_mapping.npz')
arr = data['arr_0'] 

print("Shape of array:", arr.shape)
print("First row (movie 1):", top_dense[1])
print("First row (movie 2):", top_dense[2])


Shape of array: (3000, 200)
First row (movie 1): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 3 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
First row (movie 2): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [6]:
top_k = 300
top_similar_movies = []

final_similarities = []
final_movies = []
os.makedirs('final',exist_ok=True)

for i in range(top_dense.shape[0]):
    cast_i=top_dense[i]
    similarities=[]

    for j in range (top_dense.shape[0]):
        if i==j:
            continue
        cast_j=top_dense[j]
        
        overlap_count = np.sum(np.minimum(cast_i, cast_j))
        
        if overlap_count > 0:
            similarities.append((j, overlap_count))
            if(i==1 and j==2):
                print (overlap_count)
            
    similarities_sorted = sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]

    
    # Pad with -1 if fewer than top_k
    padded = [j for j, _ in similarities_sorted] + [-1] * (top_k - len(similarities_sorted))
    top_similar_movies.append(padded)
    final_similarities.append([sim for _, sim in similarities_sorted])
    final_movies.append([j for j, _ in similarities_sorted])

# Save the results
np.savez_compressed('cast/top_300_similar_movies.npz', top_similar_movies)
np.savez_compressed('final/final_similarity.npz', np.array(final_similarities, dtype=object))
np.savez_compressed('final/final_movies.npz', np.array(final_movies, dtype=object))

9


In [7]:
loaded = np.load('cast/top_300_similar_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[0]
print(first_row)


[  20  110  137  154  176  215  257  302  325  560  648 1517 1523   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1 

In [8]:
loaded = np.load('final/final_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])
loaded = np.load('final/final_similarity.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])

[2, 4, 10, 54, 72, 86, 112, 119, 150, 151, 190, 223, 244, 250, 288, 320, 345, 364, 377, 426, 572, 589, 734, 751, 1055, 1451, 1917, 2217, 12, 953, 6, 7, 30, 97, 108, 183, 219, 331, 380, 427, 457, 463, 514, 528, 840, 961, 1678, 1956, 2067, 2500, 2668]
[np.int64(9), np.int64(6), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]


In [ ]:
[234, 434, 2727, 113, 216, 533, 976, 9, 206]
